<a href="https://colab.research.google.com/github/ShabahangShayegan/TP3_IA_autour_du_codec/blob/main/TP3_IA_autour_du_codec_etudiant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 — L'IA autour du codec : super-résolution, métriques et compromis perception-distorsion

**Cours : Machine Learning appliqué à la compression vidéo — EPITA, majeure IA — Séance 3 (3h)**

Une façon très répandue d'utiliser l'IA en compression **sans changer le codec** : on **sous-échantillonne** la vidéo avant l'encodage, puis on la **sur-échantillonne avec un réseau** après le décodage. Le codec classique code 4 fois moins de pixels ; le réseau doit reconstruire les détails.

```
1080p ──↓2──► 540p ──► x265 ──► flux ──► décodage ──► 540p ──► super-résolution ×2 ──► 1080p
1080p ─────────────────► x265 ──► flux ──► décodage ──────────────────────────────────► 1080p   (référence)
```

| Partie | Durée | Contenu |
|---|---|---|
| **A** | ~30 min | La chaîne de codage : sous-échantillonnage, x265/x264, sur-échantillonnage bicubique |
| **B** | ~20 min | Trois réseaux de super-résolution pré-entraînés : **SwinIR** (optimisé PSNR), **SwinIR-GAN** (même architecture, entraînement génératif), **Real-ESRGAN** (génératif) |
| **C** | ~60 min | **PSNR, VMAF, VMAF NEG, LPIPS** : courbes R-D, BD-rate **par métrique**, point de croisement, plan **perception-distorsion** |

Le message central : **le choix de la métrique change le classement des méthodes.**

---
**Avant de commencer**
- `Fichier > Enregistrer une copie dans Drive`, puis `Exécution > Modifier le type d'exécution > GPU (T4)`.
- Les cellules **✏️ Exercice** sont à compléter ; les **❓ Questions** vont dans le rapport.
- Les résultats sont **sauvegardés sur votre Drive** au fur et à mesure : après une coupure, ré-exécutez simplement les cellules.

## 0. Configuration, installation, ressources

In [ ]:
# ===================== Paramètres du TP =====================
ASSETS_REPO = "ShabahangShayegan/epita-ml-video-assets"   # dépôt Hugging Face (type dataset) de l'enseignant
SEQUENCES   = ["Jockey", "Bosphorus"]                       # séquences UVG (cf. TP2)
N_FRAMES    = 4                       # images par séquence (4 suffisent : le contenu change peu en 4 images)
CODEC       = "libx265"               # "libx265" ou "libx264"
QP_LR       = [27, 32, 37]            # QP pour la chaîne basse résolution (540p)
QP_HR       = [22, 27, 32, 37, 42]    # QP pour l'encodage direct en 1080p
W_HR, H_HR  = 1920, 1080              # résolution de sortie ; la chaîne code en W_HR/2 x H_HR/2
USE_DRIVE   = True
USE_PRECOMPUTED_IF_MISSING = True
RUN_VMAF, RUN_COOLCHIC = True, True
SR_LIVE     = ["SwinIR", "SwinIR-GAN", "Real-ESRGAN"]   # modèles calculés ici ; les autres viennent des résultats pré-calculés
# ============================================================
W_LR, H_LR = W_HR // 2, H_HR // 2

In [ ]:
!pip install -q spandrel lpips

In [ ]:
import os, sys, io, re, json, glob, time, shutil, subprocess, tempfile, warnings, urllib.request
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.interpolate import PchipInterpolator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "pas de GPU : la super-résolution sera lente")
torch.manual_seed(0); np.random.seed(0)
plt.rcParams["figure.dpi"] = 110
LPIPS_PRETRAINED = True

IN_COLAB = "google.colab" in sys.modules
WORK = "tp3_work"
if IN_COLAB:
    WORK = "/content/tp3_work"
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            WORK = "/content/drive/MyDrive/EPITA_TP3_sr"
        except Exception as e:
            print("Drive non monté :", e)
LOCAL = "/content/tp3_local" if IN_COLAB else os.path.join(WORK, "local")
for d in [os.path.join(WORK, s) for s in ["data", "results", "bin", "sr", "assets"]] + [LOCAL]:
    os.makedirs(d, exist_ok=True)
RES_DIR = os.path.join(WORK, "results", f"{CODEC}_{W_HR}x{H_HR}_{N_FRAMES}f")
os.makedirs(RES_DIR, exist_ok=True)
print("Dossier de travail :", WORK)

def sh(cmd, check=True, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if check and r.returncode != 0:
        raise RuntimeError(f"Échec ({r.returncode}) : {cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout + r.stderr

def hf_get(path_in_repo):
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(ASSETS_REPO, path_in_repo, repo_type="dataset", local_dir=os.path.join(WORK, "assets"))
    except Exception as e:
        print(f"  [ressource indisponible] {path_in_repo} : {type(e).__name__}")
        return None

### FFmpeg avec VMAF

Le `ffmpeg` installé dans Colab n'inclut pas **libvmaf**. On récupère une version statique qui l'intègre (préparée par l'enseignant, sinon les *builds* publics de BtbN). Elle ne sert qu'au calcul de VMAF.

In [ ]:
FFMPEG_VMAF = os.path.join(WORK, "bin", "ffmpeg_vmaf")
BTBN_URL = "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz"

def has_libvmaf(binary):
    try:
        return "libvmaf" in sh([binary, "-hide_banner", "-filters"], check=False)
    except Exception:
        return False

def setup_vmaf():
    if os.path.exists(FFMPEG_VMAF) and has_libvmaf(FFMPEG_VMAF): return True
    if has_libvmaf("ffmpeg"):
        shutil.copy(shutil.which("ffmpeg"), FFMPEG_VMAF); return True
    f = hf_get("bin/ffmpeg_vmaf")
    if f:
        shutil.copy(f, FFMPEG_VMAF); os.chmod(FFMPEG_VMAF, 0o755)
        if has_libvmaf(FFMPEG_VMAF): return True
    print("Téléchargement d'un ffmpeg statique avec libvmaf (~100 Mo)…")
    arc = os.path.join(LOCAL, "ffmpeg.tar.xz")
    sh(["wget", "-q", "-O", arc, BTBN_URL])
    sh(["tar", "-xf", arc, "-C", LOCAL])
    shutil.copy(glob.glob(os.path.join(LOCAL, "ffmpeg-*", "bin", "ffmpeg"))[0], FFMPEG_VMAF); os.chmod(FFMPEG_VMAF, 0o755)
    return has_libvmaf(FFMPEG_VMAF)

VMAF_OK = False
if RUN_VMAF:
    try:
        VMAF_OK = setup_vmaf()
    except Exception as e:
        print("VMAF indisponible :", str(e)[:300])
print("VMAF disponible :", VMAF_OK)

## 1. Données : extraits UVG 1080p (YUV 4:2:0)

Les mêmes séquences qu'au TP2. On n'utilise que les premières images de chaque séquence.

In [ ]:
UVG_URL = "https://ultravideo.fi/video/{name}_1920x1080_120fps_420_8bit_YUV_RAW.7z"
frame_bytes = lambda w, h: w * h * 3 // 2

def read_yuv420(path, w, h, n, start=0):
    fb, frames = frame_bytes(w, h), []
    with open(path, "rb") as f:
        f.seek(start * fb)
        for _ in range(n):
            buf = f.read(fb)
            if len(buf) < fb: break
            a = np.frombuffer(buf, np.uint8)
            frames.append((a[:w * h].reshape(h, w), a[w * h:w * h * 5 // 4].reshape(h // 2, w // 2), a[w * h * 5 // 4:].reshape(h // 2, w // 2)))
    return frames

def write_yuv420(path, frames):
    with open(path, "wb") as f:
        for Y, U, V in frames:
            f.write(Y.tobytes()); f.write(U.tobytes()); f.write(V.tobytes())

KR, KB = 0.2126, 0.0722   # BT.709 pleine échelle, comme aux TP1 et TP2

def yuv420_to_rgb(Y, U, V):
    y = torch.from_numpy(Y.astype(np.float32) / 255)
    u = torch.from_numpy(U.astype(np.float32) / 255 - 0.5).repeat_interleave(2, 0).repeat_interleave(2, 1)
    v = torch.from_numpy(V.astype(np.float32) / 255 - 0.5).repeat_interleave(2, 0).repeat_interleave(2, 1)
    r = y + 2 * (1 - KR) * v; b = y + 2 * (1 - KB) * u; g = (y - KR * r - KB * b) / (1 - KR - KB)
    return torch.stack([r, g, b])[None].clamp(0, 1)

def rgb_to_yuv420(x):
    x = x.detach().float().cpu().clamp(0, 1)
    r, g, b = x[0, 0], x[0, 1], x[0, 2]
    y = KR * r + (1 - KR - KB) * g + KB * b
    u = F.avg_pool2d(((b - y) / (2 * (1 - KB)) + 0.5)[None, None], 2)[0, 0]
    v = F.avg_pool2d(((r - y) / (2 * (1 - KR)) + 0.5)[None, None], 2)[0, 0]
    q = lambda t: (t.clamp(0, 1) * 255).round().byte().numpy()
    return q(y), q(u), q(v)

def synthetic_sequence(name, w, h, n):
    from skimage import data as skd
    base = [skd.astronaut, skd.chelsea, skd.rocket, skd.coffee][sum(map(ord, name)) % 4]()
    big = F.interpolate(torch.from_numpy(base).permute(2, 0, 1)[None].float() / 255, size=(h + 4 * n, w + 6 * n),
                        mode="bicubic", align_corners=False).clamp(0, 1)
    return [rgb_to_yuv420(big[..., 2 * t:2 * t + h, 3 * t:3 * t + w]) for t in range(n)]

def get_sequence(name):
    dst = os.path.join(WORK, "data", f"{name}_{W_HR}x{H_HR}_{N_FRAMES}f.yuv")
    if os.path.exists(dst) and os.path.getsize(dst) >= N_FRAMES * frame_bytes(W_HR, H_HR):
        return dst
    if (W_HR, H_HR) == (1920, 1080):
        for f in [f"uvg/{name}_1920x1080_16f.yuv", f"uvg/{name}_1920x1080_96f.yuv"]:
            src = hf_get(f)
            if src:
                write_yuv420(dst, read_yuv420(src, W_HR, H_HR, N_FRAMES)); return dst
        try:
            arc = os.path.join(LOCAL, f"{name}.7z")
            if not os.path.exists(arc):
                print(f"  téléchargement de l'archive UVG {name} (~0,7 Go)…")
                sh(["wget", "-q", "-O", arc, UVG_URL.format(name=name)])
            if shutil.which("7z") is None: sh("apt-get -qq install -y p7zip-full")
            p = subprocess.Popen(["7z", "e", "-so", arc], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
            data = p.stdout.read(N_FRAMES * frame_bytes(1920, 1080)); p.kill()
            open(dst, "wb").write(data); return dst
        except Exception as e:
            print("  site UVG indisponible :", e)
    print(f"  !!! {name} : séquence SYNTHÉTIQUE de secours (les conclusions ne valent pas pour UVG)")
    write_yuv420(dst, synthetic_sequence(name, W_HR, H_HR, N_FRAMES))
    return dst

SEQ_PATHS = {s: get_sequence(s) for s in SEQUENCES}
SEQ = {s: read_yuv420(p, W_HR, H_HR, N_FRAMES) for s, p in SEQ_PATHS.items()}
SEQ_RGB = {s: [yuv420_to_rgb(*f) for f in fr] for s, fr in SEQ.items()}
fig, axes = plt.subplots(1, len(SEQUENCES), figsize=(7 * len(SEQUENCES), 4), squeeze=False)
for ax, s in zip(axes[0], SEQUENCES):
    ax.imshow(SEQ_RGB[s][0][0].permute(1, 2, 0)); ax.set_title(s); ax.axis("off")
plt.tight_layout(); plt.show()

---
# Partie A — La chaîne de codage (~30 min)

On code la vidéo de deux façons avec le même encodeur et les mêmes réglages (QP constant, période intra 32) :
- **directe** : 1080p → codec → 1080p ;
- **basse résolution** : 1080p → **sous-échantillonnage bicubique ×2** → 540p → codec → 540p → **sur-échantillonnage** → 1080p.

Le débit est toujours exprimé **par rapport aux pixels de la sortie 1080p** (bpp), pour que les deux chaînes soient comparables. À titre indicatif : 0,01 bpp en 1080p à 30 images/s ≈ 620 kbit/s.

In [ ]:
def downscale(src, dst, n):
    # filtre bicubique avec anti-repliement (celui d'ffmpeg), sur le YUV 4:2:0
    sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{W_HR}x{H_HR}", "-i", src,
        "-frames:v", str(n), "-vf", f"scale={W_LR}:{H_LR}:flags=bicubic", "-f", "rawvideo", "-pix_fmt", "yuv420p", dst])

def encode_decode(src, w, h, qp, out_yuv):
    # Encode puis décode un fichier YUV ; renvoie (octets du flux, temps d'encodage en s)
    bs = out_yuv + (".hevc" if CODEC == "libx265" else ".h264")
    if CODEC == "libx265":
        enc = ["-c:v", "libx265", "-preset", "medium", "-x265-params", f"qp={qp}:keyint=32:min-keyint=32:scenecut=0:log-level=error"]
    else:
        enc = ["-c:v", "libx264", "-preset", "medium", "-qp", str(qp), "-g", "32", "-keyint_min", "32", "-sc_threshold", "0"]
    t0 = time.time()
    sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w}x{h}", "-r", "30",
        "-i", src, "-frames:v", str(N_FRAMES), *enc, bs])
    t_enc = time.time() - t0
    sh(["ffmpeg", "-y", "-loglevel", "error", "-i", bs, "-f", "rawvideo", "-pix_fmt", "yuv420p", out_yuv])
    n = os.path.getsize(bs); os.remove(bs)
    return n, t_enc

LR_DIR = os.path.join(LOCAL, "lr"); os.makedirs(LR_DIR, exist_ok=True)
LR_SRC = {}
for s in SEQUENCES:
    LR_SRC[s] = os.path.join(LR_DIR, f"{s}_lr_src.yuv")
    if not os.path.exists(LR_SRC[s]): downscale(SEQ_PATHS[s], LR_SRC[s], N_FRAMES)
print("Sources basse résolution prêtes :", {s: f"{W_LR}x{H_LR}" for s in SEQUENCES})

### Métriques

- **PSNR YUV** (6:1:1), comme au TP2 : fidélité pixel à pixel.
- **LPIPS** (Zhang et al., CVPR 2018) : distance entre les **caractéristiques profondes** d'un réseau (AlexNet) des deux images, calibrée sur des jugements humains. **Plus c'est bas, mieux c'est.**
- **VMAF** (Netflix) : fusion par SVM de caractéristiques (fidélité visuelle, détail, mouvement), entraînée sur des notes subjectives de vidéos compressées et redimensionnées. **VMAF NEG** (*No Enhancement Gain*) neutralise le gain obtenu par simple rehaussement de contraste ou de netteté.

### ✏️ Exercice 1 — LPIPS d'une image

`lpips_model` attend deux tenseurs `N×3×H×W` **dans $[-1, 1]$**. Complétez `lpips_rgb(a, b)` pour deux tenseurs RGB `1×3×H×W` dans $[0, 1]$ ; renvoyez un flottant.

In [ ]:
import lpips
lpips_model = lpips.LPIPS(net="alex", pretrained=True, pnet_rand=not LPIPS_PRETRAINED, verbose=False).eval().to(device)

def lpips_rgb(a, b):
    # TODO : ramener a et b dans [-1, 1], les envoyer sur `device`, appeler lpips_model sans gradient
    raise NotImplementedError("Exercice 1")

In [ ]:
def psnr_plane(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 100.0 if mse == 0 else min(100.0, 10 * np.log10(255.0 ** 2 / mse))

def psnr_yuv(src, rec):
    return (6 * psnr_plane(src[0], rec[0]) + psnr_plane(src[1], rec[1]) + psnr_plane(src[2], rec[2])) / 8

def vmaf_scores(ref_path, dist_path):
    # VMAF et VMAF NEG moyens sur la séquence (None si indisponible).
    # Un passage par modèle : la syntaxe à deux modèles (« name=… ») dépend trop de la version de ffmpeg.
    if not VMAF_OK: return None, None
    out = []
    for model in ["vmaf_v0.6.1", "vmaf_v0.6.1neg"]:
        log = os.path.join(LOCAL, f"{model}.json")
        sh([FFMPEG_VMAF, "-y", "-loglevel", "error",
            "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{W_HR}x{H_HR}", "-i", dist_path,
            "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{W_HR}x{H_HR}", "-i", ref_path,
            "-lavfi", f"[0:v][1:v]libvmaf=model=version={model}:log_fmt=json:log_path={log}:n_threads={os.cpu_count()}",
            "-f", "null", "-"])
        out.append(json.load(open(log))["pooled_metrics"]["vmaf"]["mean"])
    return out[0], out[1]

def evaluate(seq, rec_frames_yuv, rec_path):
    # rec_frames_yuv : liste de (Y, U, V) 1080p ; rec_path : le même contenu écrit sur disque (pour VMAF)
    src = SEQ[seq]
    p = np.mean([psnr_yuv(s, r) for s, r in zip(src, rec_frames_yuv)])
    l = np.mean([lpips_rgb(SEQ_RGB[seq][i], yuv420_to_rgb(*r)) for i, r in enumerate(rec_frames_yuv)])
    v, vn = vmaf_scores(SEQ_PATHS[seq], rec_path)
    return dict(psnr=float(p), lpips=float(l), vmaf=v, vmaf_neg=vn)

In [ ]:
def res_file(pipeline, seq, qp): return os.path.join(RES_DIR, f"{pipeline}_{seq}_{qp}.json")
def save(pipeline, seq, qp, d):
    d.update(pipeline=pipeline, seq=seq, qp=qp); json.dump(d, open(res_file(pipeline, seq, qp), "w")); return d

# 1) Encodage direct en 1080p
for s in SEQUENCES:
    for qp in QP_HR:
        if os.path.exists(res_file("direct 1080p", s, qp)): continue
        rec = os.path.join(LOCAL, "rec_direct.yuv")
        n, t = encode_decode(SEQ_PATHS[s], W_HR, H_HR, qp, rec)
        d = save("direct 1080p", s, qp, dict(bpp=8 * n / (N_FRAMES * W_HR * H_HR), t_enc=t / N_FRAMES,
                 **evaluate(s, read_yuv420(rec, W_HR, H_HR, N_FRAMES), rec)))
        print(f"direct {s} QP{qp}: {d['bpp']:.4f} bpp, PSNR {d['psnr']:.2f}, LPIPS {d['lpips']:.3f}, VMAF {d['vmaf']}")

# 2) Chaîne basse résolution : encodage 540p (les flux décodés sont gardés pour la partie B)
LR_DEC, LR_BYTES = {}, {}
for s in SEQUENCES:
    for qp in QP_LR + [None]:                          # None : 540p non compressé (super-résolution seule)
        out = os.path.join(LR_DIR, f"{s}_lr_qp{qp}.yuv")
        meta = out + ".json"
        if qp is None:
            LR_DEC[(s, qp)], LR_BYTES[(s, qp)] = LR_SRC[s], 0; continue
        if not os.path.exists(meta):
            n, t = encode_decode(LR_SRC[s], W_LR, H_LR, qp, out); json.dump(dict(n=n, t=t), open(meta, "w"))
        LR_DEC[(s, qp)], LR_BYTES[(s, qp)] = out, json.load(open(meta))["n"]
print("Flux basse résolution prêts.")

### Sur-échantillonnage bicubique (la référence sans IA)

In [ ]:
def upscale_bicubic(x_lr):
    return F.interpolate(x_lr, scale_factor=2, mode="bicubic", align_corners=False).clamp(0, 1)

def run_sr_pipeline(name, upscale_fn):
    tim = []
    for s in SEQUENCES:
        for qp in QP_LR + [None]:
            key = "nocomp" if qp is None else qp
            if os.path.exists(res_file(name, s, key)): continue
            frames_lr = read_yuv420(LR_DEC[(s, qp)], W_LR, H_LR, N_FRAMES)
            out_frames = []
            for f in frames_lr:
                x = yuv420_to_rgb(*f).to(device)
                if device == "cuda": torch.cuda.synchronize()
                t0 = time.time()
                with torch.no_grad():
                    y = upscale_fn(x)
                if device == "cuda": torch.cuda.synchronize()
                tim.append(time.time() - t0)
                out_frames.append(rgb_to_yuv420(y[..., :H_HR, :W_HR]))
            rec = os.path.join(LOCAL, "rec_sr.yuv"); write_yuv420(rec, out_frames)
            d = save(name, s, key, dict(bpp=8 * LR_BYTES[(s, qp)] / (N_FRAMES * W_HR * H_HR),
                                        t_sr=float(np.mean(tim[-len(frames_lr):])), **evaluate(s, out_frames, rec)))
            print(f"{name} {s} QP{qp}: {d['bpp']:.4f} bpp, PSNR {d['psnr']:.2f}, LPIPS {d['lpips']:.3f}, VMAF {d['vmaf']}")

run_sr_pipeline("540p + bicubique", upscale_bicubic)

**❓ Questions — Partie A**
1. Pour un même QP, combien de bits la chaîne 540p économise-t-elle par rapport au 1080p direct ? Pourquoi pas exactement 4 fois moins ?
2. Le cas « non compressé » (540p sans codage, puis bicubique) donne la **meilleure qualité atteignable** par la chaîne bicubique. Que vaut ce plafond en PSNR ? Que dit-il sur l'intérêt de la chaîne à haut débit ?

---
# Partie B — Super-résolution par réseaux pré-entraînés (~20 min)

On charge les poids publics avec **spandrel**, une bibliothèque qui reconnaît automatiquement des dizaines d'architectures de restauration d'images (utilisée notamment par ComfyUI et chaiNNer). Trois modèles ×2 :

| Modèle | Architecture | Entraînement | Dégradations vues à l'entraînement |
|---|---|---|---|
| **SwinIR** (classique) | Transformer à fenêtres glissantes | perte L1 → **maximise le PSNR** | sous-échantillonnage bicubique propre |
| **SwinIR-GAN** (« real-world ») | **la même** | L1 + perceptuelle + **adversariale** | flou, bruit, **compression JPEG** (BSRGAN) |
| **Real-ESRGAN x2plus** | CNN (RRDB) | L1 + perceptuelle + **adversariale** | dégradations synthétiques réalistes, dont **compression** |

La comparaison SwinIR / SwinIR-GAN isole l'effet de **la fonction de coût et des données**, à architecture égale.

⏱️ Environ 5 minutes de calcul sur T4 pour les trois modèles. Chaque image est traitée **en une seule passe** (et en demi-précision quand le modèle le permet) ; le découpage en tuiles n'est utilisé qu'en cas de mémoire GPU insuffisante.

In [ ]:
from spandrel import ModelLoader

SR_MODELS = {
    "SwinIR":      ("001_classicalSR_DF2K_s64w8_SwinIR-M_x2.pth", "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x2.pth"),
    "SwinIR-GAN":  ("003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x2_GAN.pth", "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x2_GAN.pth"),
    "Real-ESRGAN": ("RealESRGAN_x2plus.pth", "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"),
}

def get_weights(fname, url):
    dst = os.path.join(WORK, "sr", fname)
    if os.path.exists(dst): return dst
    f = hf_get(f"sr/{fname}")
    if f: shutil.copy(f, dst); return dst
    try:
        urllib.request.urlretrieve(url, dst); return dst
    except Exception as e:
        print(f"  poids {fname} introuvables : {e}"); return None

def load_sr(name):
    path = get_weights(*SR_MODELS[name])
    if path is None: return None
    m = ModelLoader().load_from_file(path).to(device).eval()
    if device == "cuda" and m.supports_half:
        m.half()                                   # demi-précision : 2 à 3 fois plus rapide sur T4
    dt = next(m.model.parameters()).dtype
    print(f"{name}: architecture {m.architecture.name}, ×{m.scale}, {sum(p.numel() for p in m.model.parameters())/1e6:.1f} M paramètres, {dt}")
    return m

def upscale_sr(model, x):
    # Une seule passe sur l'image entière ; tuiles seulement si la mémoire GPU ne suffit pas
    dt = next(model.model.parameters()).dtype
    try:
        return model(x.to(dt)).float().clamp(0, 1)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return tiled(model, x, tile=512)

def tiled(model, x, tile=256, overlap=16):
    # Super-résolution par tuiles avec recouvrement ; les bords de chaque tuile sont jetés
    _, _, h, w = x.shape; s = model.scale
    out = torch.zeros(1, 3, h * s, w * s, device=x.device)
    step = tile - 2 * overlap
    for i0 in range(0, h, step):
        for j0 in range(0, w, step):
            a, b = max(i0 - overlap, 0), max(j0 - overlap, 0)
            c, d = min(i0 + step + overlap, h), min(j0 + step + overlap, w)
            y = model(x[..., a:c, b:d].to(next(model.model.parameters()).dtype)).float()
            ia, ja = (i0 - a) * s, (j0 - b) * s
            ih, jw = min(step, h - i0) * s, min(step, w - j0) * s
            out[..., i0 * s:i0 * s + ih, j0 * s:j0 * s + jw] = y[..., ia:ia + ih, ja:ja + jw]
    return out.clamp(0, 1)

sr_loaded = []
for name in SR_MODELS:
    if all(os.path.exists(res_file(name, s, q)) for s in SEQUENCES for q in QP_LR + ["nocomp"]):
        sr_loaded.append(name); continue
    if name not in SR_LIVE:
        print(f"{name} : non calculé ici (résultats pré-calculés utilisés en partie C)"); continue
    model = load_sr(name)
    if model is None: continue
    t0 = time.time()
    run_sr_pipeline(name, lambda x, m=model: upscale_sr(m, x))
    print(f"  → {name} : {time.time() - t0:.0f} s")
    sr_loaded.append(name)
    del model
    if device == "cuda": torch.cuda.empty_cache()

In [ ]:
# Comparaison visuelle sur un détail, à QP 32 en 540p
s, qp = SEQUENCES[0], 32
f_lr = read_yuv420(LR_DEC[(s, qp)], W_LR, H_LR, 1)[0]
x_lr = yuv420_to_rgb(*f_lr).to(device)
y0, x0, hh, ww = H_HR // 3, W_HR // 3, H_HR // 4, W_HR // 5
crops = {"original 1080p": SEQ_RGB[s][0][0], "540p + bicubique": upscale_bicubic(x_lr)[0].cpu()}
for name in sr_loaded:
    m = load_sr(name)
    with torch.no_grad():
        crops[name] = upscale_sr(m, x_lr)[0].cpu()
    del m
rec = os.path.join(LOCAL, "rec_direct_vis.yuv")
encode_decode(SEQ_PATHS[s], W_HR, H_HR, 37, rec)
crops["1080p direct QP37"] = yuv420_to_rgb(*read_yuv420(rec, W_HR, H_HR, 1)[0])[0]
fig, axes = plt.subplots(1, len(crops), figsize=(3.6 * len(crops), 3.4))
for ax, (k, v) in zip(axes, crops.items()):
    ax.imshow(v[:, y0:y0 + hh, x0:x0 + ww].permute(1, 2, 0).numpy()); ax.set_title(k, fontsize=9); ax.axis("off")
plt.suptitle(f"{s}, image 0 — chaîne 540p à QP {qp}"); plt.tight_layout(); plt.show()

**❓ Questions — Partie B**
3. Décrivez visuellement chaque sortie. Que fait SwinIR (classique) des **artefacts de compression** du flux 540p ? Pourquoi, vu ses données d'entraînement ?
4. Les sorties génératives semblent-elles plus nettes ? Les détails ajoutés sont-ils **vrais** (présents dans l'original) ? Trouvez un exemple de détail inventé.

---
# Partie C — Le choix de la métrique change le classement (~60 min)

In [ ]:
rows = [json.load(open(p)) for p in glob.glob(os.path.join(RES_DIR, "*.json"))]
if USE_PRECOMPUTED_IF_MISSING:
    have = {(r["pipeline"], r["seq"]) for r in rows}
    f = hf_get(f"precomputed/tp3_{CODEC}.jsonl") if (W_HR, H_HR, N_FRAMES) == (1920, 1080, 4) else None
    if f:
        for line in open(f):
            r = json.loads(line)
            if r["seq"] in SEQUENCES and (r["pipeline"], r["seq"]) not in have:
                r["pipeline"] += " (pré-calculé)"; rows.append(r)
df = pd.DataFrame(rows)
df_rd = df[df.qp != "nocomp"].copy(); df_rd["qp"] = df_rd.qp.astype(int)
METRICS = [("psnr", "PSNR YUV (dB) ↑", True), ("lpips", "LPIPS ↓", False)]
if df.vmaf.notna().any():
    METRICS += [("vmaf", "VMAF ↑", True), ("vmaf_neg", "VMAF NEG ↑", True)]
print("Chaînes :", sorted(df.pipeline.unique()), "| métriques :", [m[0] for m in METRICS])

def style(p):
    return "k--" if p.startswith("direct") else {"540p + bicubique": "s:", "SwinIR": "o-", "SwinIR-GAN": "^-", "Real-ESRGAN": "D-"}.get(p.split(" (")[0], "x-")

for s in SEQUENCES:
    fig, axes = plt.subplots(1, len(METRICS), figsize=(5.2 * len(METRICS), 4.2), squeeze=False)
    for ax, (m, lab, _) in zip(axes[0], METRICS):
        for p, g in df_rd[df_rd.seq == s].groupby("pipeline"):
            g = g.sort_values("bpp"); ax.plot(g.bpp, g[m], style(p), label=p, ms=4)
        ax.set_xscale("log"); ax.set_xlabel("bpp (pixels 1080p, échelle log)"); ax.set_ylabel(lab); ax.grid(alpha=.3, which="both")
    axes[0, 0].legend(fontsize=7); plt.suptitle(f"{s} — {CODEC}, {N_FRAMES} images"); plt.tight_layout(); plt.show()

### ✏️ Exercice 2 — BD-rate pour toutes les métriques

La fonction `bd_rate` (TP1, TP2) suppose que **la qualité augmente** avec le débit. Complétez `bd_table(metric, higher_is_better)` : pour chaque chaîne et chaque séquence, BD-rate par rapport à l'**encodage direct 1080p**, puis moyenne sur les séquences. Pour LPIPS (plus bas = meilleur), utilisez $-\text{LPIPS}$ comme qualité.

In [ ]:
def bd_rate(rate_anchor, q_anchor, rate_test, q_test):
    def interp(rate, q):
        q, lr = np.asarray(q, float), np.log(np.asarray(rate, float)); i = np.argsort(q)
        return PchipInterpolator(q[i], lr[i])
    lo, hi = max(min(q_anchor), min(q_test)), min(max(q_anchor), max(q_test))
    if hi <= lo: return np.nan
    fa, ft = interp(rate_anchor, q_anchor), interp(rate_test, q_test)
    return (np.exp((ft.integrate(lo, hi) - fa.integrate(lo, hi)) / (hi - lo)) - 1) * 100

In [ ]:
def bd_table(metric, higher_is_better):
    sign = 1 if higher_is_better else -1
    out = {}
    for p in df_rd.pipeline.unique():
        if p.startswith("direct"): continue
        # TODO : pour chaque séquence, ancre = lignes "direct 1080p", test = lignes de la chaîne p ;
        #        BD-rate avec la qualité multipliée par `sign` ; out[p] = moyenne (np.nanmean) sur les séquences
        raise NotImplementedError("Exercice 2")
    return pd.Series(out, name=metric)

In [ ]:
bd = pd.concat([bd_table(m, hib) for m, _, hib in METRICS], axis=1).round(1)
print("BD-rate (%) par rapport à l'encodage direct 1080p — négatif = la chaîne avec super-résolution économise du débit")
display(bd)
print("\nClassement par métrique (1 = meilleure chaîne) :")
display(bd.rank(axis=0).astype("Int64"))

### ✏️ Exercice 3 — Point de croisement

En général, la chaîne basse résolution gagne **à bas débit** (le codec gaspille moins de bits sur des détails qu'il ne peut de toute façon pas coder) et perd **à haut débit** (la résolution réduite plafonne la qualité). Complétez `crossover(...)` : interpolez la qualité de chaque chaîne en fonction de $\log(\text{bpp})$ (`np.interp` sur les points triés) sur une grille commune, et renvoyez le **plus grand débit** pour lequel la chaîne SR est au moins aussi bonne que le direct (`None` s'il n'y en a pas).

In [ ]:
def crossover(seq, pipeline, metric, higher_is_better=True):
    a = df_rd[(df_rd.pipeline.str.startswith("direct")) & (df_rd.seq == seq)].sort_values("bpp")
    t = df_rd[(df_rd.pipeline == pipeline) & (df_rd.seq == seq)].sort_values("bpp")
    # TODO : plage de débits commune, grille log, interpolation des deux courbes,
    #        plus grand débit où la chaîne `pipeline` est au moins aussi bonne
    raise NotImplementedError("Exercice 3")

In [ ]:
tab = []
for s in SEQUENCES:
    for p in [p for p in df_rd.pipeline.unique() if not p.startswith("direct")]:
        row = dict(seq=s, chaîne=p)
        for m, _, hib in METRICS:
            c = crossover(s, p, m, hib)
            row[m] = "—" if c is None else f"{c:.4f} bpp (~{c * W_HR * H_HR * 30 / 1e3:.0f} kbit/s)"
        tab.append(row)
print("Débit maximal jusqu'auquel la chaîne avec super-résolution est au moins aussi bonne que le direct (à 30 images/s) :")
display(pd.DataFrame(tab).set_index(["seq", "chaîne"]))

### Le plan perception-distorsion

Blau et Michaeli (CVPR 2018) ont montré qu'on ne peut pas minimiser simultanément la **distorsion** (écart pixel à pixel, ici le PSNR) et maximiser la **qualité perceptuelle** (proximité avec la distribution des images naturelles) : il existe une frontière. Plaçons nos méthodes dans ce plan, sans compression puis à QP 32.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, key in zip(axes, ["nocomp", 32]):
    sub = df[(df.qp.astype(str) == str(key)) & (~df.pipeline.str.startswith("direct"))]
    for p, g in sub.groupby("pipeline"):
        mk = {"540p + bicubique": "s", "SwinIR": "o", "SwinIR-GAN": "^", "Real-ESRGAN": "D"}.get(p.split(" (")[0], "x")
        ax.scatter(g.psnr, g.lpips, s=60, marker=mk, label=p)
    ax.invert_yaxis(); ax.set_xlabel("PSNR YUV (dB) → moins de distorsion"); ax.set_ylabel("LPIPS (axe inversé) → meilleure perception")
    ax.set_title("540p non compressé" if key == "nocomp" else "540p codé à QP 32"); ax.grid(alpha=.3)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

if "vmaf_neg" in [m[0] for m in METRICS]:
    g = df[df.qp.astype(str) != "nocomp"].groupby("pipeline")[["vmaf", "vmaf_neg"]].mean()
    g["écart VMAF - VMAF NEG"] = g.vmaf - g.vmaf_neg
    print("Gain « d'amélioration » détecté par VMAF NEG :"); display(g.round(2))

t = df.dropna(subset=["t_sr"]).groupby("pipeline").t_sr.mean().mul(1000).round(1) if "t_sr" in df else pd.Series(dtype=float)
print(f"\nTemps de super-résolution ({'GPU ' + torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}), ms par image 1080p :"); display(t)

**❓ Questions — Partie C**
5. Reportez le tableau des **classements par métrique**. Quelle chaîne gagne en PSNR ? En LPIPS ? En VMAF ? Expliquez chaque victoire par la fonction de coût du modèle.
6. Comparez **SwinIR** et **SwinIR-GAN** (même architecture). Dans le plan perception-distorsion, dans quel sens l'entraînement adversarial déplace-t-il le modèle ? Est-ce cohérent avec le théorème de Blau et Michaeli ?
7. Que mesure l'écart **VMAF − VMAF NEG** ? Quelle chaîne en profite le plus, et pourquoi un fournisseur de streaming devrait-il s'en méfier ?
8. Jusqu'à quel débit la chaîne avec super-résolution vaut-elle la peine (exercice 3) ? Le point de croisement dépend-il de la métrique et du contenu (Jockey vs Bosphorus) ? Faites le lien avec l'**échelle de débits (*bitrate ladder*)** adaptée au contenu (*per-title encoding*).
9. Relancez la partie A avec `CODEC = "libx264"` (changez la cellule de paramètres) : le gain de la super-résolution est-il plus ou moins important qu'avec x265 ? Pourquoi ?
10. Le temps de super-résolution est-il compatible avec un décodage temps réel sur un téléviseur ou un téléphone ? Citez une solution industrielle de super-résolution côté client et ce qu'elle sacrifie.
11. Le cours a présenté le **filtrage in-loop appris** (JVET NNVC). En quoi est-ce différent d'un post-traitement comme ici ? Pourquoi un filtre in-loop doit-il être **déterministe au bit près** entre encodeur et décodeur ?

---
## Pour aller plus loin
- **Codage pour les machines** (MPEG VCM / FCM) : refaites la partie C en remplaçant les métriques par la **précision d'un détecteur d'objets** sur les images décodées. Le classement change-t-il encore ?
- Ajoutez votre propre modèle de super-résolution : spandrel en charge des dizaines (HAT, DAT, SPAN…).

## Rendu
Rapport d'équipe : courbes R-D par métrique, tableaux BD-rate et classements, points de croisement, plan perception-distorsion, réponses argumentées aux questions 1 à 11.